In [29]:
import pandas as pd

# 1. Individua le proteine di AD, PD e in comune
AD_prots = set(pd.read_csv('../files/networks/AD_nodes.csv')['name'].to_list())
PD_prots = set(pd.read_csv('../files/networks/PD_nodes.csv')['name'].to_list())
common_prots = AD_prots.intersection(PD_prots)

# 2. Annota la matrice di similarità con la colonna label
sim_matrix = pd.read_csv('/home/developer/Documents/Tesi/files/output/similarity_matrix_all_mean_cleaned.csv')
sim_matrix = sim_matrix.rename(columns={"Unnamed: 0": "name"})
sim_matrix['features'] = sim_matrix.iloc[:, 1:].values.tolist()
sim_matrix = sim_matrix[['name', 'features']]
sim_matrix['label'] = sim_matrix.iloc[:, 0].apply(lambda x: 2 if x in common_prots else (0 if x in AD_prots else 1))
sim_matrix.to_csv('test.csv', index_label='id')

sim_matrix.to_csv('./pre_processing_output/classification_three_labels_nodes_sim.csv', index_label='id')

In [31]:
# Da qui costruisco un dataset dove le proteine in comune sono duplicate per staccare le componenti connesse di AD e PD e 
#   le componenti connesse minori vengono escluse.
#   Per farlo mi baso su i csv prodotti con il pre processing sugli embedding, sostituendo i vettori di similarità a questi.

nodes_df = pd.read_csv('./pre_processing_output/duplicated_nodes_main_components.csv')

# Duplico i record di sim_matrix aggiungendo _AD e _PD alle proteine in comune e imposto i label alla malattia corrispondente
sim_matrix_dup = sim_matrix.copy()
sim_matrix_dup[sim_matrix_dup['label'] == 2]

df_ad = sim_matrix_dup[sim_matrix_dup['label'] == 2].copy()
df_ad['name'] += '_AD'
df_ad['label'] = 0

df_pd = sim_matrix_dup[sim_matrix_dup['label'] == 2].copy()
df_pd['name'] += '_PD'
df_pd['label'] = 1

sim_matrix_dup = sim_matrix_dup[sim_matrix_dup['label'] != 2]
sim_matrix_dup = pd.concat([sim_matrix_dup, df_ad, df_pd], ignore_index=True)

name_to_sim = dict(zip(sim_matrix_dup['name'], sim_matrix_dup['features']))
nodes_df['GO_embeddings'] = nodes_df['STRING_id'].map(name_to_sim)
nodes_df.to_csv('./pre_processing_output/duplicated_nodes_main_components_sim.csv')